In [6]:
import csv
import logging
import os
import sys
import time
from dataclasses import dataclass, fields
from datetime import datetime
from io import StringIO
from pathlib import Path
from typing import Optional

import requests
from bs4 import BeautifulSoup

# ╔══════════════════════════════════════════════════════╗
# ║                     CONFIG                          ║
# ║  Set SOURCE_URL  — OR —  set USE_MOCK = True        ║
# ╚══════════════════════════════════════════════════════╝

SOURCE_URL  = ""                        # e.g. "https://example-news.com"
USE_MOCK    = True                      # True = use built-in sample data
OUTPUT_FILE = "/kaggle/working/headlines.csv"   # change path for Colab if needed

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ── Constants ─────────────────────────────────────────────────────────────────
REQUEST_TIMEOUT = 10
REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
RETRY_ATTEMPTS = 3
RETRY_DELAY    = 2

# ── Mock HTML (used when USE_MOCK = True) ─────────────────────────────────────
MOCK_HTML = """
<html><body>
  <article class="news-item">
    <h2 class="headline"><a href="/article/1" class="story-link">Python 4.0 Released with Major Performance Improvements</a></h2>
    <span class="author">By Jane Smith</span>
    <span class="timestamp">2024-06-15 08:30</span>
    <p class="summary">The Python Software Foundation announced Python 4.0 today, bringing significant speed boosts and new syntax features.</p>
    <span class="category">Technology</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/2" class="story-link">AI Chip Demand Drives Record Semiconductor Revenue</a></h2>
    <span class="author">By Tom Richards</span>
    <span class="timestamp">2024-06-15 09:15</span>
    <p class="summary">Global semiconductor revenues hit an all-time high as AI workloads fuel unprecedented demand for specialized chips.</p>
    <span class="category">Business</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/3" class="story-link">Open Source LLM Outperforms Proprietary Models on Benchmarks</a></h2>
    <span class="author">By Priya Nair</span>
    <span class="timestamp">2024-06-15 10:00</span>
    <p class="summary">A new open-source language model has surpassed several leading commercial models in reasoning and coding tasks.</p>
    <span class="category">AI</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/4" class="story-link">Cybersecurity Breaches Cost Businesses $9.4 Trillion in 2024</a></h2>
    <span class="author">By David Okonkwo</span>
    <span class="timestamp">2024-06-15 11:45</span>
    <p class="summary">A new report reveals a sharp increase in breach-related costs, urging businesses to adopt zero-trust architectures.</p>
    <span class="category">Security</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/5" class="story-link">Quantum Computing Startup Achieves 1000-Qubit Milestone</a></h2>
    <span class="author">By Sofia Mendes</span>
    <span class="timestamp">2024-06-15 13:20</span>
    <p class="summary">A Silicon Valley startup crossed the 1000-qubit threshold with record-low error rates.</p>
    <span class="category">Science</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/6" class="story-link">EU Passes Sweeping New Data Privacy Regulations</a></h2>
    <span class="author">By Lena Fischer</span>
    <span class="timestamp">2024-06-15 14:05</span>
    <p class="summary">The EU approved landmark legislation expanding citizen data rights and imposing stricter penalties on tech giants.</p>
    <span class="category">Policy</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/7" class="story-link">Remote Work Tools Market Expected to Reach $96B by 2027</a></h2>
    <span class="author">By Marcus Lee</span>
    <span class="timestamp">2024-06-15 15:30</span>
    <p class="summary">Analysts project explosive growth in collaboration software as hybrid work becomes the global standard.</p>
    <span class="category">Business</span>
  </article>
  <article class="news-item">
    <h2 class="headline"><a href="/article/8" class="story-link">Electric Vehicle Battery Range Doubles with New Solid-State Tech</a></h2>
    <span class="author">By Aisha Bakare</span>
    <span class="timestamp">2024-06-15 16:10</span>
    <p class="summary">A manufacturer unveiled solid-state cell tech that doubles EV range and charges in under 10 minutes.</p>
    <span class="category">Technology</span>
  </article>
</body></html>
"""

In [7]:
# ── Data model ────────────────────────────────────────────────────────────────
@dataclass
class Article:
    headline  : str
    author    : str
    timestamp : str
    summary   : str
    category  : str
    url       : str
    scraped_at: str



def fetch_html(url: str) -> Optional[str]:
    log.info(f"Fetching → {url}")
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            r = requests.get(url, headers=REQUEST_HEADERS, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            log.info(f"  ✔  HTTP {r.status_code} (attempt {attempt})")
            return r.text
        except requests.exceptions.Timeout:
            log.warning(f"  Attempt {attempt} timed out")
        except requests.exceptions.ConnectionError as e:
            log.warning(f"  Attempt {attempt} connection error: {e}")
        except requests.exceptions.HTTPError as e:
            log.error(f"  HTTP error: {e}"); return None
        except requests.exceptions.RequestException as e:
            log.error(f"  Request error: {e}"); return None

        if attempt < RETRY_ATTEMPTS:
            log.info(f"  Retrying in {RETRY_DELAY}s …")
            time.sleep(RETRY_DELAY)

    log.error(f"  ✖  All {RETRY_ATTEMPTS} attempts failed")
    return None



def _text(tag, selector: str, prefix: str = "") -> str:
    el = tag.select_one(selector)
    if not el:
        return "N/A"
    text = el.get_text(strip=True)
    if prefix and text.startswith(prefix):
        text = text[len(prefix):]
    return text or "N/A"


def parse_articles(html: str, base_url: str = "") -> list[Article]:
    soup       = BeautifulSoup(html, "html.parser")
    scraped_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    articles   = []

    cards = soup.select("article.news-item")
    log.info(f"  Found {len(cards)} article cards")

    for i, card in enumerate(cards, 1):
        try:
            link_tag = card.select_one(".story-link")
            headline = link_tag.get_text(strip=True) if link_tag else "N/A"
            href     = link_tag.get("href", "") if link_tag else ""
            full_url = (base_url.rstrip("/") + href) if href.startswith("/") else href

            articles.append(Article(
                headline  = headline,
                author    = _text(card, ".author",    prefix="By "),
                timestamp = _text(card, ".timestamp"),
                summary   = _text(card, ".summary"),
                category  = _text(card, ".category"),
                url       = full_url,
                scraped_at= scraped_at,
            ))
            log.info(f"  [{i:02d}] {headline[:65]}")
        except Exception as e:
            log.warning(f"  Skipped card {i}: {e}")

    return articles



def save_to_csv(articles: list[Article], output_path: str) -> bool:
    if not articles:
        log.warning("No articles to save."); return False

    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        suffix = datetime.now().strftime("%Y%m%d_%H%M%S")
        path   = path.with_name(f"{path.stem}_{suffix}{path.suffix}")
        log.info(f"File exists — saving as {path.name}")

    col_names = [f.name for f in fields(Article)]
    try:
        with open(path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=col_names)
            writer.writeheader()
            for a in articles:
                writer.writerow({f.name: getattr(a, f.name) for f in fields(Article)})
        log.info(f"  ✔  Saved {len(articles)} articles → {path}  ({path.stat().st_size/1024:.1f} KB)")
        return True
    except (PermissionError, OSError) as e:
        log.error(f"  ✖  Could not write file: {e}"); return False


# REPORT 
def print_report(articles: list[Article]) -> None:
    from collections import Counter
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  SCRAPE REPORT  —  {len(articles)} articles extracted")
    print(sep)

    cats = Counter(a.category for a in articles)
    print("\n  Categories:")
    for cat, count in cats.most_common():
        print(f"    {cat:<15} {'▪' * count}  ({count})")

    print("\n  Headlines:")
    for i, a in enumerate(articles, 1):
        print(f"    {i:02d}. [{a.category}] {a.headline}")
        print(f"        {a.author}  ·  {a.timestamp}")
    print(f"\n{sep}\n")

In [8]:
# MAIN 
def main():
    print("\n" + "=" * 60)
    print("  NEWS WEB SCRAPER")
    print("=" * 60)

    
    if USE_MOCK:
        log.info("Using built-in mock data (USE_MOCK = True)")
        html     = MOCK_HTML
        base_url = ""
    elif SOURCE_URL:
        html     = fetch_html(SOURCE_URL)
        base_url = SOURCE_URL
        if html is None:
            log.error("Could not retrieve HTML. Exiting."); return
    else:
        log.error("Set SOURCE_URL or set USE_MOCK = True in the CONFIG block.")
        return

    
    print("\n  Parsing HTML …")
    articles = parse_articles(html, base_url=base_url)
    if not articles:
        log.warning("No articles found. Check your CSS selectors."); return

    
    print("\n  Saving to CSV …")
    save_to_csv(articles, OUTPUT_FILE)

    
    print_report(articles)


main()   

21:28:32  INFO      Using built-in mock data (USE_MOCK = True)
21:28:32  INFO        Found 8 article cards
21:28:32  INFO        [01] Python 4.0 Released with Major Performance Improvements
21:28:32  INFO        [02] AI Chip Demand Drives Record Semiconductor Revenue
21:28:32  INFO        [03] Open Source LLM Outperforms Proprietary Models on Benchmarks
21:28:32  INFO        [04] Cybersecurity Breaches Cost Businesses $9.4 Trillion in 2024
21:28:32  INFO        [05] Quantum Computing Startup Achieves 1000-Qubit Milestone
21:28:32  INFO        [06] EU Passes Sweeping New Data Privacy Regulations
21:28:32  INFO        [07] Remote Work Tools Market Expected to Reach $96B by 2027
21:28:32  INFO        [08] Electric Vehicle Battery Range Doubles with New Solid-State Tech
21:28:32  INFO      File exists — saving as headlines_20260616_212832.csv
21:28:32  INFO        ✔  Saved 8 articles → /kaggle/working/headlines_20260616_212832.csv  (1.9 KB)



  NEWS WEB SCRAPER

  Parsing HTML …

  Saving to CSV …

  SCRAPE REPORT  —  8 articles extracted

  Categories:
    Technology      ▪▪  (2)
    Business        ▪▪  (2)
    AI              ▪  (1)
    Security        ▪  (1)
    Science         ▪  (1)
    Policy          ▪  (1)

  Headlines:
    01. [Technology] Python 4.0 Released with Major Performance Improvements
        Jane Smith  ·  2024-06-15 08:30
    02. [Business] AI Chip Demand Drives Record Semiconductor Revenue
        Tom Richards  ·  2024-06-15 09:15
    03. [AI] Open Source LLM Outperforms Proprietary Models on Benchmarks
        Priya Nair  ·  2024-06-15 10:00
    04. [Security] Cybersecurity Breaches Cost Businesses $9.4 Trillion in 2024
        David Okonkwo  ·  2024-06-15 11:45
    05. [Science] Quantum Computing Startup Achieves 1000-Qubit Milestone
        Sofia Mendes  ·  2024-06-15 13:20
    06. [Policy] EU Passes Sweeping New Data Privacy Regulations
        Lena Fischer  ·  2024-06-15 14:05
    07. [Business] 